# W3 timing smoke (design check, ~5 minutes)

Times the candidate denoisers for W3 ("Does the loop know where it is?") on a T4: training
steps at batch 128 with fp16 autocast, and forward passes for the probe harvest. Random
tensors only, so no dataset is needed and nothing is kept but the timings.

**Run:** Runtime → Change runtime type → **T4 GPU** → Run all. Allow the Drive mount.
The result lands in `MyDrive/semcore/w3/` as `timing_<stamp>.json`; the last cell also
prints the table. Safe to re-run; each run writes a new file.


In [ ]:
# ── Setup: GPU check, Drive mount ──
import json, subprocess, time
from pathlib import Path
import torch
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NONE DETECTED')
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU.'
from google.colab import drive
drive.mount('/content/drive')
SEM = Path('/content/drive/MyDrive/semcore')
assert SEM.exists(), 'MyDrive/semcore not found: mounted the right Google account?'
OUT = SEM / 'w3'
OUT.mkdir(exist_ok=True)
print('torch', torch.__version__, '| results ->', OUT)


In [ ]:
# ── Model classes, verbatim from windows/power/w3_budget_check.py ──
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def sinusoid(t, dim):
    half = dim // 2
    f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    a = t.float()[:, None] * f[None]
    return torch.cat([a.sin(), a.cos()], dim=1)


class Res(nn.Module):
    def __init__(self, a, b, tdim):
        super().__init__()
        self.n1, self.c1 = nn.GroupNorm(8, a), nn.Conv2d(a, b, 3, 1, 1)
        self.t = nn.Linear(tdim, b) if tdim else None
        self.n2, self.c2 = nn.GroupNorm(8, b), nn.Conv2d(b, b, 3, 1, 1)
        self.skip = nn.Conv2d(a, b, 1) if a != b else nn.Identity()

    def forward(self, x, temb):
        h = self.c1(F.silu(self.n1(x)))
        if self.t is not None:
            h = h + self.t(temb)[:, :, None, None]
        h = self.c2(F.silu(self.n2(h)))
        return h + self.skip(x)


class Attn(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n, self.qkv, self.o = nn.GroupNorm(8, c), nn.Conv2d(c, 3 * c, 1), nn.Conv2d(c, c, 1)

    def forward(self, x, _=None):
        b, c, h, w = x.shape
        q, k, v = self.qkv(self.n(x)).reshape(b, 3, c, h * w).unbind(1)
        a = torch.softmax(q.transpose(1, 2) @ k / math.sqrt(c), dim=-1)
        return x + self.o((v @ a.transpose(1, 2)).reshape(b, c, h, w))


class UNet(nn.Module):
    def __init__(self, cin=1, C=64, mult=(1, 2, 2), clock=True):
        super().__init__()
        tdim = 4 * C if clock else 0
        self.C, self.clock = C, clock
        self.temb = nn.Sequential(nn.Linear(C, tdim), nn.SiLU(), nn.Linear(tdim, tdim)) if clock else None
        self.inp = nn.Conv2d(cin, C, 3, 1, 1)
        chs, self.down, ch = [C], nn.ModuleList(), C
        for i, m in enumerate(mult):
            for _ in range(2):
                blk = nn.ModuleList([Res(ch, C * m, tdim)])
                ch = C * m
                if i == len(mult) - 1:
                    blk.append(Attn(ch))
                self.down.append(blk)
                chs.append(ch)
            if i < len(mult) - 1:
                self.down.append(nn.ModuleList([nn.Conv2d(ch, ch, 3, 2, 1)]))
                chs.append(ch)
        self.mid = nn.ModuleList([Res(ch, ch, tdim), Attn(ch), Res(ch, ch, tdim)])
        self.up = nn.ModuleList()
        for i, m in reversed(list(enumerate(mult))):
            for _ in range(3):
                blk = nn.ModuleList([Res(ch + chs.pop(), C * m, tdim)])
                ch = C * m
                if i == len(mult) - 1:
                    blk.append(Attn(ch))
                self.up.append(blk)
            if i > 0:
                self.up.append(nn.ModuleList([nn.Upsample(scale_factor=2), nn.Conv2d(ch, ch, 3, 1, 1)]))
        self.out = nn.Sequential(nn.GroupNorm(8, ch), nn.SiLU(), nn.Conv2d(ch, cin, 3, 1, 1))

    def forward(self, x, t):
        temb = self.temb(sinusoid(t, self.C)) if self.clock else None
        h, hs = self.inp(x), []
        hs.append(h)
        for blk in self.down:
            if isinstance(blk[0], nn.Conv2d):
                h = blk[0](h)
            else:
                for layer in blk:
                    h = layer(h, temb)
            hs.append(h)
        for layer in self.mid:
            h = layer(h, temb)
        for blk in self.up:
            if isinstance(blk[0], nn.Upsample):
                h = blk[1](blk[0](h))
            else:
                h = torch.cat([h, hs.pop()], dim=1)
                for layer in blk:
                    h = layer(h, temb)
        return self.out(h)


In [ ]:
# ── Timing: train steps (fp16 autocast, AdamW) and forward passes ──
BATCH, WARM, STEPS = 128, 15, 60
dev = 'cuda'
rows = []
for cin, data in ((1, 'MNIST'), (3, 'CIFAR-10')):
    for C in (32, 48, 64):
        for clock in ((True, False) if C == 64 else (True,)):
            torch.manual_seed(0)
            m = UNet(cin=cin, C=C, clock=clock).to(dev)
            opt = torch.optim.AdamW(m.parameters(), lr=2e-4)
            scaler = torch.amp.GradScaler('cuda')
            x = torch.randn(BATCH, cin, 32, 32, device=dev)
            t = torch.randint(0, 1000, (BATCH,), device=dev)
            torch.cuda.reset_peak_memory_stats()
            for i in range(WARM + STEPS):
                if i == WARM:
                    torch.cuda.synchronize(); t0 = time.time()
                with torch.autocast('cuda', dtype=torch.float16):
                    loss = F.mse_loss(m(x, t), x)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt); scaler.update()
            torch.cuda.synchronize()
            train_s = (time.time() - t0) / STEPS
            m.eval()
            with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
                for i in range(WARM + STEPS):
                    if i == WARM:
                        torch.cuda.synchronize(); t0 = time.time()
                    m(x, t)
            torch.cuda.synchronize()
            fwd_s = (time.time() - t0) / STEPS
            n_train = 60000 if cin == 1 else 50000
            row = {'data': data, 'C': C, 'clock': clock,
                   'params_M': round(sum(p.numel() for p in m.parameters()) / 1e6, 2),
                   'train_s_per_step': round(train_s, 4),
                   'epochs_per_hour': round(3600 / (train_s * n_train / BATCH), 1),
                   'fwd_s_per_batch': round(fwd_s, 4),
                   'peak_mem_GB': round(torch.cuda.max_memory_allocated() / 1e9, 2)}
            rows.append(row)
            print(row)
            del m, opt
            torch.cuda.empty_cache()
stamp = time.strftime('%Y%m%d_%H%M%S')
rec = {'gpu': gpu.stdout.strip(), 'torch': torch.__version__, 'batch': BATCH,
       'steps_timed': STEPS, 'rows': rows}
(OUT / f'timing_{stamp}.json').write_text(json.dumps(rec, indent=1))
print(f'\nSaved MyDrive/semcore/w3/timing_{stamp}.json')
print(f"\n{'data':9s} {'C':>3s} {'clock':>5s} {'params':>7s} {'s/step':>7s} {'ep/h':>6s} {'fwd s':>7s} {'mem':>5s}")
for r in rows:
    print(f"{r['data']:9s} {r['C']:3d} {str(r['clock']):>5s} {r['params_M']:6.1f}M {r['train_s_per_step']:7.3f} "
          f"{r['epochs_per_hour']:6.1f} {r['fwd_s_per_batch']:7.4f} {r['peak_mem_GB']:5.2f}")
